## 1. Setup & hyperparameters

In [42]:
import os
import random
import numpy as np

random.seed(42)
np.random.seed(42)

# --- keep everything tiny so training/testing is fast on CPU with plain NumPy ---
NUM_SAMPLES = 100
MAX_ARTICLE_LEN = 200
MAX_SUMMARY_LEN = 50
EMB_DIM = 16              # embedding size
HID_DIM = 32              # RNN hidden size
EPOCHS = 100
LEARNING_RATE = 0.01
GRAD_CLIP = 5.0
TRAIN_FRACTION = 0.8


## 2. Load a small subset of the BBC News Summary dataset

In [43]:
DATA_DIR = os.path.join("..", "BBC News Summary")
ARTICLES_DIR = os.path.join(DATA_DIR, "News Articles")
SUMMARIES_DIR = os.path.join(DATA_DIR, "Summaries")

def load_all_pairs():
    pairs = []
    for category in sorted(os.listdir(ARTICLES_DIR)):
        art_dir = os.path.join(ARTICLES_DIR, category)
        sum_dir = os.path.join(SUMMARIES_DIR, category)
        if not os.path.isdir(art_dir):
            continue
        for fname in sorted(os.listdir(art_dir)):
            art_path = os.path.join(art_dir, fname)
            sum_path = os.path.join(sum_dir, fname)
            if not os.path.exists(sum_path):
                continue
            with open(art_path, encoding="latin-1") as f:
                lines = f.read().strip().split("\n")
            article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
            with open(sum_path, encoding="latin-1") as f:
                summary = f.read().strip()

            article_words = article.lower().split()[:MAX_ARTICLE_LEN]
            summary_words = summary.lower().split()[:MAX_SUMMARY_LEN]
            if len(article_words) < 5 or len(summary_words) < 3:
                continue
            pairs.append((" ".join(article_words), " ".join(summary_words)))
    return pairs

all_pairs = load_all_pairs()
random.shuffle(all_pairs)
pairs = all_pairs[:NUM_SAMPLES]

split = int(TRAIN_FRACTION * len(pairs))
train_pairs = pairs[:split]
test_pairs = pairs[split:]

print(f"Total usable pairs in dataset: {len(all_pairs)}")
print(f"Using {len(pairs)} pairs -> {len(train_pairs)} train / {len(test_pairs)} test")
print("\nExample pair:")
print("Article: ", train_pairs[0][0])
print("Summary: ", train_pairs[0][1])


Total usable pairs in dataset: 2225
Using 100 pairs -> 80 train / 20 test

Example pair:
Article:  the uk property market remains robust despite the recent slowdown, according to mortgage lender bradford & bingley and housebuilder george wimpey. b&b said the buy-to-let market - in which the bank is a major player - would continue to grow much faster than the wider mortgage market. the comments came as it reported a 6% rise in profits to â£280.2m ($532m). wimpey reported a 19% rise in profits to â£450.7m and said recent new home reservations were better than expected. recent housing market surveys have indicated that the uk property market has cooled in recent months after several years of rapid growth. last week, figures from the council of mortgage lenders (cml) indicated that the popularity of buy-to-let mortgages - a key phenomenon of the housing boom - could be waning. but b&b - which has a 22% share of the uk buy-to-let mortgage market - said that while rates of growth were modera

## 3. Build the vocabulary and encode text as integer ids

In [44]:
PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
SPECIAL_TOKENS = [PAD, SOS, EOS, UNK]

vocab_words = set()
for article, summary in pairs:
    vocab_words.update(article.split())
    vocab_words.update(summary.split())

itos = SPECIAL_TOKENS + sorted(vocab_words)
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)
print(f"Vocabulary size: {VOCAB_SIZE}")

def encode(text, add_sos=False, add_eos=False):
    ids = [stoi[SOS]] if add_sos else []
    ids += [stoi.get(w, stoi[UNK]) for w in text.split()]
    if add_eos:
        ids.append(stoi[EOS])
    return np.array(ids, dtype=np.int64)

train_data = [(encode(a), encode(b, add_sos=True, add_eos=True)) for a, b in train_pairs]
test_data = [(encode(a), encode(b, add_sos=True, add_eos=True)) for a, b in test_pairs]


Vocabulary size: 6036


## 4. NumPy building blocks (activations, softmax, weight init)

In [45]:
def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def xavier(shape):
    fan_in = shape[1] if len(shape) > 1 else shape[0]
    fan_out = shape[0]
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, size=shape)


## 5. Model parameters

A plain dictionary holds every learnable array. Keeping parameters in a dict (rather
than a class) makes the forward/backward functions below simple, stateless functions
that just take `params` as an argument.

In [46]:
def init_params():
    p = {}
    # encoder
    p["Emb_enc"] = xavier((VOCAB_SIZE, EMB_DIM))
    p["Wxh_enc"] = xavier((HID_DIM, EMB_DIM))
    p["Whh_enc"] = xavier((HID_DIM, HID_DIM))
    p["bh_enc"] = np.zeros(HID_DIM)
    # Bahdanau attention: score(s_prev, h_i) = va . tanh(Wa @ s_prev + Ua @ h_i)
    p["Wa"] = xavier((HID_DIM, HID_DIM))
    p["Ua"] = xavier((HID_DIM, HID_DIM))
    p["va"] = xavier((HID_DIM,))
    # decoder (input = [prev token embedding ; attention context])
    p["Emb_dec"] = xavier((VOCAB_SIZE, EMB_DIM))
    p["Wxh_dec"] = xavier((HID_DIM, EMB_DIM + HID_DIM))
    p["Whh_dec"] = xavier((HID_DIM, HID_DIM))
    p["bh_dec"] = np.zeros(HID_DIM)
    # output projection to vocabulary logits
    p["Why"] = xavier((VOCAB_SIZE, HID_DIM))
    p["by"] = np.zeros(VOCAB_SIZE)
    return p

params = init_params()
print("Parameter shapes:")
for k, v in params.items():
    print(f"  {k:10s} {v.shape}")


Parameter shapes:
  Emb_enc    (6036, 16)
  Wxh_enc    (32, 16)
  Whh_enc    (32, 32)
  bh_enc     (32,)
  Wa         (32, 32)
  Ua         (32, 32)
  va         (32,)
  Emb_dec    (6036, 16)
  Wxh_dec    (32, 48)
  Whh_dec    (32, 32)
  bh_dec     (32,)
  Why        (6036, 32)
  by         (6036,)


## 6. Encoder forward pass

A vanilla (tanh) RNN reads the article token by token and returns **every** hidden
state `H` (shape `[T_x, HID_DIM]`) — the decoder's attention mechanism needs access to
all of them, not just the final one.

In [47]:
def encoder_forward(x_ids, p):
    T = len(x_ids)
    H = np.zeros((T, HID_DIM))
    X = np.zeros((T, EMB_DIM))          # cached embeddings, needed for backward
    h_prev = np.zeros(HID_DIM)
    for t in range(T):
        x_t = p["Emb_enc"][x_ids[t]]
        z_t = p["Wxh_enc"] @ x_t + p["Whh_enc"] @ h_prev + p["bh_enc"]
        h_t = np.tanh(z_t)
        X[t] = x_t
        H[t] = h_t
        h_prev = h_t
    cache = {"x_ids": x_ids, "X": X, "H": H}
    return H, cache


## 7. Bahdanau (additive) attention

Given the decoder's previous hidden state `s_prev` and all encoder hidden states `H`,
compute attention weights `alpha` over the source positions and the resulting context
vector (a weighted sum of `H`).

In [48]:
def attention(s_prev, H, p):
    # U[i] = tanh(Wa @ s_prev + Ua @ H[i])   -- shape [T_x, HID_DIM]
    U = np.tanh(H @ p["Ua"].T + s_prev @ p["Wa"].T)
    scores = U @ p["va"]                     # shape [T_x]
    alpha = softmax(scores)                  # attention weights, shape [T_x]
    context = alpha @ H                      # shape [HID_DIM]
    cache = {"U": U, "alpha": alpha, "H": H, "s_prev": s_prev}
    return context, alpha, cache


## 8. Decoder forward pass (single time step)

At each step the decoder: (1) attends over the encoder states, (2) feeds
`[prev-token-embedding ; context]` through a tanh RNN cell, (3) projects the new
hidden state to vocabulary logits.

In [49]:
def decoder_step(y_prev_id, s_prev, H, p):
    context, alpha, attn_cache = attention(s_prev, H, p)
    e_t = p["Emb_dec"][y_prev_id]
    x_t = np.concatenate([e_t, context])
    z_t = p["Wxh_dec"] @ x_t + p["Whh_dec"] @ s_prev + p["bh_dec"]
    s_t = np.tanh(z_t)
    logits = p["Why"] @ s_t + p["by"]
    cache = {"y_prev_id": y_prev_id, "e_t": e_t, "x_t": x_t, "s_prev": s_prev,
             "s_t": s_t, "context": context, "attn": attn_cache}
    return logits, s_t, alpha, cache


## 9. Full forward pass over one (article, summary) pair

Runs the encoder once, then unrolls the decoder with **teacher forcing** (the true
previous summary token is fed as input at every step), accumulating the
cross-entropy loss. All per-step caches are kept for the backward pass.

In [50]:
def forward_pass(x_ids, y_ids, p):
    H, enc_cache = encoder_forward(x_ids, p)
    s_prev = H[-1]                     # bridge: decoder starts from encoder's last state
    loss = 0.0
    dec_caches = []
    for t in range(len(y_ids) - 1):
        y_prev_id = y_ids[t]
        y_target_id = y_ids[t + 1]
        logits, s_t, alpha, cache = decoder_step(y_prev_id, s_prev, H, p)
        probs = softmax(logits)
        loss += -np.log(probs[y_target_id] + 1e-9)
        cache["probs"] = probs
        cache["y_target_id"] = y_target_id
        dec_caches.append(cache)
        s_prev = s_t
    loss /= max(1, len(y_ids) - 1)
    return loss, enc_cache, dec_caches


## 10. Backward pass — decoder & attention

Standard backprop-through-time over the decoder steps (in reverse), plus the extra
gradient paths introduced by attention: the context vector and the attention "keys"
(`Ua @ H[i]`) both send gradient back into the encoder hidden states `H`. Those are
accumulated into `dH` and handed off to the encoder's backward pass.

In [51]:
def zeros_like_params(p):
    return {k: np.zeros_like(v) for k, v in p.items()}

def decoder_backward(x_ids, dec_caches, p):
    T_x = len(x_ids)
    grads = zeros_like_params(p)
    dH = np.zeros((T_x, HID_DIM))     # gradient flowing into encoder hidden states
    ds_next = np.zeros(HID_DIM)       # gradient w.r.t. s_t coming from the future step

    n_steps = len(dec_caches)
    for t in reversed(range(n_steps)):
        c = dec_caches[t]

        # --- output projection ---
        dlogits = c["probs"].copy()
        dlogits[c["y_target_id"]] -= 1.0
        dlogits /= n_steps
        grads["Why"] += np.outer(dlogits, c["s_t"])
        grads["by"] += dlogits
        ds_t = p["Why"].T @ dlogits + ds_next

        # --- decoder RNN cell ---
        dz_t = ds_t * (1 - c["s_t"] ** 2)
        grads["Wxh_dec"] += np.outer(dz_t, c["x_t"])
        grads["Whh_dec"] += np.outer(dz_t, c["s_prev"])
        grads["bh_dec"] += dz_t
        dx_t = p["Wxh_dec"].T @ dz_t
        de_t, dcontext = dx_t[:EMB_DIM], dx_t[EMB_DIM:]
        ds_prev = p["Whh_dec"].T @ dz_t

        grads["Emb_dec"][c["y_prev_id"]] += de_t

        # --- attention ---
        ac = c["attn"]
        alpha, U, H = ac["alpha"], ac["U"], ac["H"]
        dalpha = H @ dcontext
        dH += np.outer(alpha, dcontext)                       # context path
        dscores = alpha * (dalpha - np.sum(alpha * dalpha))   # softmax backward
        grads["va"] += dscores @ U
        dU = np.outer(dscores, p["va"])
        dz_u = dU * (1 - U ** 2)                               # tanh backward
        grads["Wa"] += np.outer(dz_u.sum(axis=0), c["s_prev"])
        grads["Ua"] += dz_u.T @ H
        ds_prev += p["Wa"].T @ dz_u.sum(axis=0)
        dH += dz_u @ p["Ua"]                                   # attention-key path

        ds_next = ds_prev

    return grads, dH, ds_next  # ds_next here is the gradient w.r.t. the encoder's last state


## 11. Backward pass — encoder (BPTT)

The encoder receives two kinds of gradient at each position `i`: the recurrent
gradient from step `i+1`, and the direct gradient `dH[i]` sent by attention (every
decoder step looked at every encoder position). The very last encoder step also
receives the gradient from being used as the decoder's initial hidden state.

In [52]:
def encoder_backward(enc_cache, dH, ds0, p):
    x_ids, X, H = enc_cache["x_ids"], enc_cache["X"], enc_cache["H"]
    T = len(x_ids)
    grads = zeros_like_params(p)
    dh_next = ds0
    for t in reversed(range(T)):
        dh_total = dH[t] + dh_next
        dz = dh_total * (1 - H[t] ** 2)
        h_prev = H[t - 1] if t > 0 else np.zeros(HID_DIM)
        grads["Wxh_enc"] += np.outer(dz, X[t])
        grads["Whh_enc"] += np.outer(dz, h_prev)
        grads["bh_enc"] += dz
        dx = p["Wxh_enc"].T @ dz
        grads["Emb_enc"][x_ids[t]] += dx
        dh_next = p["Whh_enc"].T @ dz
    return grads

def backward_pass(x_ids, enc_cache, dec_caches, p):
    dec_grads, dH, ds0 = decoder_backward(x_ids, dec_caches, p)
    enc_grads = encoder_backward(enc_cache, dH, ds0, p)
    grads = dec_grads
    for k, v in enc_grads.items():
        grads[k] += v
    return grads


## 12. Gradient clipping + Adam optimizer

In [53]:
def clip_grads(grads, max_norm=GRAD_CLIP):
    for k in grads:
        norm = np.linalg.norm(grads[k])
        if norm > max_norm:
            grads[k] *= max_norm / (norm + 1e-9)
    return grads

class Adam:
    def __init__(self, params, lr=LEARNING_RATE, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * (grads[k] ** 2)
            m_hat = self.m[k] / (1 - self.beta1 ** self.t)
            v_hat = self.v[k] / (1 - self.beta2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


## 13. Training loop

Because the vocabulary, hidden size and sequences are all tiny, a few hundred epochs
over a few dozen examples trains in seconds on plain NumPy.

In [54]:
params = init_params()
optimizer = Adam(params)

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train_data)
    total_loss = 0.0
    for x_ids, y_ids in train_data:
        loss, enc_cache, dec_caches = forward_pass(x_ids, y_ids, params)
        grads = backward_pass(x_ids, enc_cache, dec_caches, params)
        grads = clip_grads(grads)
        optimizer.step(params, grads)
        total_loss += loss
    if epoch == 1 or epoch % 20 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  avg loss: {total_loss / len(train_data):.4f}")


Epoch   1/100  avg loss: 8.0511
Epoch  20/100  avg loss: 2.2636
Epoch  40/100  avg loss: 0.7258
Epoch  60/100  avg loss: 0.3248
Epoch  80/100  avg loss: 0.8967
Epoch 100/100  avg loss: 0.1926


## 14. Greedy inference

Encode the article, then decode one token at a time by always picking the
highest-probability next word, feeding it back in as input, until `<EOS>` or a
maximum length is reached.

In [55]:
def summarize(article_text, max_len=MAX_SUMMARY_LEN + 2):
    x_ids = encode(article_text)
    H, _ = encoder_forward(x_ids, params)
    s_prev = H[-1]
    y_id = stoi[SOS]
    words = []
    for _ in range(max_len):
        logits, s_t, alpha, _ = decoder_step(y_id, s_prev, H, params)
        y_id = int(np.argmax(logits))
        if itos[y_id] == EOS:
            break
        words.append(itos[y_id])
        s_prev = s_t
    return " ".join(words)


## 15. Evaluate on held-out test articles

In [56]:
print("=" * 60)
print("TEST ON HELD-OUT BBC NEWS ARTICLES")
print("=" * 60)
for article, reference in test_pairs:
    generated = summarize(article)
    print(f"\nArticle:   {article}")
    print(f"Reference: {reference}")
    print(f"Generated: {generated}")


TEST ON HELD-OUT BBC NEWS ARTICLES

Article:   michael howard has dismissed fears from some conservatives that his plans for â£4bn tax cuts are too modest. he defended the package, saying it was a plan for the tories' first budget and he hoped to be able to go further. the tories on monday highlighted â£35bn in "wasteful" spending they would stop to allow tax cuts, reduced borrowing and more spending on key services. labour and the liberal democrats say the party's sums do not add up and claim it would cut frontline services. the tory tax plan follows complaints from some of the party's mps that mr howard and shadow chancellor oliver letwin have taken too long to unveil the proposals. now they have promised a figure but have yet to reveal which taxes would be targeted. tory backbencher edward leigh said the proposals were a step in the right direction but he told the financial times: "i would come up sooner with much greater tax cuts." interviewed on bbc radio 2's jeremy vine show, mr 